In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Turn off the GPU, use the CPU instead
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [2]:
# Parse .tfrecords files
# Image stored as raw bytes and label stored as int (0, 1, 2)
feature_description = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

def parse_example(example_proto):
    example = tf.io.parse_single_example(example_proto, feature_description)
    image = tf.io.decode_raw(example["image"], tf.uint8)
    image = tf.reshape(image, [299, 299, 1])
    image = tf.cast(image, tf.float32) / 255.0  # normalize
    label = tf.cast(example["label"], tf.int32)
    return image, label

# Load the dataset
train_tfrecord = "Kaggle_Data/Correct_Training_Data/training10_0.tfrecords" 

train_ds = tf.data.TFRecordDataset(train_tfrecord)
train_ds = train_ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)

# Inspect samples
for img, lbl in train_ds.take(3):
    print("Image shape:", img.shape, "Label:", lbl.numpy())

# Dataset size and labeling
labels = []
for _, lbl in train_ds:
    labels.append(lbl.numpy())

labels = np.array(labels)
unique, counts = np.unique(labels, return_counts=True)

print("\nClass distribution:")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c} samples")
print("Total samples:", len(labels))

I0000 00:00:1765977054.084156  191457 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1765977054.142807  191457 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1765977055.656639  191457 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
E0000 00:00:1765977056.723505  191457 cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ER

Image shape: (299, 299, 1) Label: 0
Image shape: (299, 299, 1) Label: 0
Image shape: (299, 299, 1) Label: 0

Class distribution:
Class 0: 9698 samples
Class 1: 822 samples
Class 2: 657 samples
Total samples: 11177


I0000 00:00:1765977058.123598  191457 local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [3]:
# Training pipeline
BATCH_SIZE = 16
SHUFFLE_BUFFER = 1000

train_ds = tf.data.TFRecordDataset(train_tfrecord)
train_ds = train_ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = (
    train_ds
    .shuffle(SHUFFLE_BUFFER)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Verify dimensions
for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)

Batch image shape: (16, 299, 299, 1)
Batch label shape: (16,)


In [4]:
# Model architecture
model = models.Sequential([
    layers.Input(shape=(299, 299, 1)),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),

    layers.Dense(3, activation="softmax")
])

# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=losses.SparseCategoricalCrossentropy(),
    metrics=[
        metrics.SparseCategoricalAccuracy(name="accuracy")
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 299, 299, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 299, 299, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 149, 149, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 149, 149, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 149, 149, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 74, 74, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 74, 74, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 74, 74, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 37, 37, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 37, 37, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 18, 18, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,531 (1.49 MB)

 Trainable params: 389,571 (1.49 MB)

 Non-trainable params: 960 (3.75 KB)

In [6]:
# Recompute training labels as NumPy array
train_labels = []

for _, lbl in tf.data.TFRecordDataset(train_tfrecord).map(parse_example):
    train_labels.append(lbl.numpy())

train_labels = np.array(train_labels)

print("Training label distribution:")
unique, counts = np.unique(train_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

# Compute class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_labels
)

class_weights = dict(enumerate(class_weights))
print("\nClass weights:", class_weights)

# Load validation data from .npy files
x_val = np.load("Kaggle_Data/Correct_Validation_Data/validation_data.npy")
y_val = np.load("Kaggle_Data/Correct_Validation_Data/validation_labels.npy")

# Verify channel dimension
if x_val.ndim == 3:
    x_val = x_val[..., np.newaxis]

x_val = x_val.astype("float32") / 255.0
y_val = y_val.astype("int32")

print("\nValidation images shape:", x_val.shape)
print("Validation labels shape:", y_val.shape)

# Train model
EPOCHS = 5

history = model.fit(
    train_ds,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    class_weight=class_weights
)

I0000 00:00:1765977123.620127  191457 local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Training label distribution:
Class 0: 9698
Class 1: 822
Class 2: 657

Class weights: {0: np.float64(0.3841685570908091), 1: np.float64(4.532441200324412), 2: np.float64(5.670725520040588)}

Validation images shape: (7682, 299, 299, 1)
Validation labels shape: (7682,)
Epoch 1/5
    699/Unknown 386s 549ms/step - accuracy: 0.4849 - loss: 1.2574

/home/fg/tf-gpu/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


699/699 ━━━━━━━━━━━━━━━━━━━━ 451s 641ms/step - accuracy: 0.5434 - loss: 1.1748 - val_accuracy: 0.7404 - val_loss: 0.6375
Epoch 2/5
699/699 ━━━━━━━━━━━━━━━━━━━━ 463s 662ms/step - accuracy: 0.6048 - loss: 1.0329 - val_accuracy: 0.3560 - val_loss: 2.1945
Epoch 3/5
699/699 ━━━━━━━━━━━━━━━━━━━━ 468s 669ms/step - accuracy: 0.6429 - loss: 0.9640 - val_accuracy: 0.6490 - val_loss: 0.8473
Epoch 4/5
699/699 ━━━━━━━━━━━━━━━━━━━━ 0s 587ms/step - accuracy: 0.6429 - loss: 0.9361

I0000 00:00:1765978950.564877  191775 local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


699/699 ━━━━━━━━━━━━━━━━━━━━ 507s 676ms/step - accuracy: 0.6471 - loss: 0.9351 - val_accuracy: 0.6612 - val_loss: 0.7547
Epoch 5/5
699/699 ━━━━━━━━━━━━━━━━━━━━ 495s 709ms/step - accuracy: 0.6523 - loss: 0.9060 - val_accuracy: 0.2611 - val_loss: 2.0296


In [7]:
# Make predictions on validation set
y_pred_probs = model.predict(x_val)
y_pred = np.argmax(y_pred_probs, axis=1)

# Generate confusion matrix
cm = confusion_matrix(y_val, y_pred)
print("Confusion matrix:\n", cm)

# Generate normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print("\nNormalized confusion matrix:\n", np.round(cm_norm, 2))

# Report per-class metrics
report = classification_report(y_val, y_pred, target_names=["Normal", "Benign", "Malignant"])
print("\nClassification report:\n", report)

# Report per-class accuracy manually
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Accuracy for class {i} ({['Normal','Benign','Malignant'][i]}): {acc:.2f}")

241/241 ━━━━━━━━━━━━━━━━━━━━ 59s 244ms/step
Confusion matrix:
 [[1472 3457 1763]
 [   0  210  373]
 [   0   83  324]]

Normalized confusion matrix:
 [[0.22 0.52 0.26]
 [0.   0.36 0.64]
 [0.   0.2  0.8 ]]

Classification report:
               precision    recall  f1-score   support

      Normal       1.00      0.22      0.36      6692
      Benign       0.06      0.36      0.10       583
   Malignant       0.13      0.80      0.23       407

    accuracy                           0.26      7682
   macro avg       0.40      0.46      0.23      7682
weighted avg       0.88      0.26      0.33      7682

Accuracy for class 0 (Normal): 0.22
Accuracy for class 1 (Benign): 0.36
Accuracy for class 2 (Malignant): 0.80
